# Real-Time Price Comparison Agent: Grounding LLMs with Live Singapore E-Commerce Data

Large language models carry product knowledge from their training data — but prices change daily, new models launch, and promotions come and go. **Without access to a live data source, any price the model gives you is almost certainly wrong.**

This recipe demonstrates a concrete, practical solution: connect an OpenAI model to the [BuyWhere](https://buywhere.ai) real-time Singapore product catalog API so that every price answer is grounded in live data from Harvey Norman, Shopee, Lazada, and Qoo10.

## What you will learn

| Concept | What the recipe shows |
|---|---|
| **The stale-data problem** | Ask the model a price question without tools — watch it guess or refuse |
| **Real-time grounding** | Wire up BuyWhere as an OpenAI function-calling tool to get live prices |
| **Structured price comparison** | Use `response_format` JSON output to produce a machine-readable comparison table |
| **Multi-product batch comparison** | Compare several products in a single agent turn using parallel tool calls |
| **Best-price extraction** | Automatically identify the cheapest retailer for any query |

> **Disclosure:** BuyWhere submitted this recipe idea. The API is independent of OpenAI. BuyWhere offers free developer API keys with no credit card required.

## Prerequisites

### 1. OpenAI API key
Create one at [platform.openai.com/api-keys](https://platform.openai.com/api-keys).

### 2. BuyWhere API key (free)
Generate a free developer key at [buywhere.ai/api-keys](https://buywhere.ai/api-keys).  
Free tier: **1,000 queries/month · 10 requests/minute** — plenty for this recipe.

### 3. `.env` file

```
OPENAI_API_KEY=sk-...
BUYWHERE_API_KEY=bw_live_...
```

## 1. Install and import

In [ ]:
%pip install openai requests python-dotenv --quiet

In [ ]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
BUYWHERE_API_KEY = os.environ["BUYWHERE_API_KEY"]

# Correct base URL from BuyWhere OpenAPI spec (note: .io not .ai)
BUYWHERE_BASE_URL = "https://api.buywhere.io/v1"
BUYWHERE_HEADERS = {"Authorization": f"Bearer {BUYWHERE_API_KEY}"}

MODEL = "gpt-4o"
print("Ready.")

## 2. The stale-knowledge problem

Let's first ask the model a direct price question **without any tools**. This illustrates why tool use matters for commerce use cases.

In [ ]:
question = "What is the current price of the iPhone 16 Pro 256GB in Singapore? Which retailer is cheapest?"

response_no_tools = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful shopping assistant."},
        {"role": "user", "content": question},
    ],
    # No tools — model must rely on training data alone
)

print("Without tools (stale training data):")
print("-" * 50)
print(response_no_tools.choices[0].message.content)

The model will either:
- Give a price from its training data (which may be months or years out of date), or
- Correctly refuse and admit it doesn't have up-to-date information.

Either way, the answer cannot be trusted for an actual purchasing decision. Let's fix that.

## 3. Wire up BuyWhere as a function-calling tool

We register `search_singapore_products` as an OpenAI tool. The model will call it whenever it needs live price data.

In [ ]:
def search_singapore_products(query: str, limit: int = 10) -> dict:
    """
    Search real-time product prices in Singapore across Harvey Norman,
    Shopee, Lazada, Qoo10, and other retailers via the BuyWhere API.

    Args:
        query: Product name or description (e.g. 'iPhone 16 Pro 256GB').
        limit: Max results to return (1-100, default 10).

    Returns:
        Live product listings with prices, retailer names, and product IDs.
    """
    response = requests.get(
        f"{BUYWHERE_BASE_URL}/search",
        headers=BUYWHERE_HEADERS,
        params={"q": query, "limit": limit},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


def get_best_price_singapore(query: str) -> dict:
    """
    Find the single lowest-priced listing for a product in Singapore.

    Args:
        query: Product name or description.

    Returns:
        The cheapest available listing with price and retailer.
    """
    response = requests.get(
        f"{BUYWHERE_BASE_URL}/products/best-price",
        headers=BUYWHERE_HEADERS,
        params={"q": query},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_singapore_products",
            "description": (
                "Search real-time product prices in Singapore from Harvey Norman, Shopee, Lazada, "
                "Qoo10, and other retailers. Use this to get live prices before answering any "
                "price or availability question."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Product name or description, e.g. 'iPhone 16 Pro 256GB' or 'Samsung 55 inch 4K TV'."
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Number of results to return (1-100). Default 10.",
                        "default": 10
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_best_price_singapore",
            "description": (
                "Find the single cheapest price for a product in Singapore across all retailers. "
                "Use this when the user explicitly asks for the lowest price or cheapest option."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Product name or description."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

TOOL_DISPATCH = {
    "search_singapore_products": search_singapore_products,
    "get_best_price_singapore": get_best_price_singapore,
}

print("Tools registered.")

## 4. Agent loop with tool execution

A minimal but complete function-calling loop:  
model → tool call → result → model → final answer.

In [ ]:
SYSTEM_PROMPT = """\
You are a Singapore price comparison assistant.
You MUST call the search tools before answering any question about product prices or availability.
Never rely on your training knowledge for prices — always fetch live data first.
Present prices in SGD. Be concise and lead with the most actionable finding.
"""


def run_price_agent(user_message: str, response_format: dict | None = None) -> str:
    """
    Run the price comparison agent for a single query.

    Args:
        user_message: The user's price or product question.
        response_format: Optional structured output format (e.g. JSON schema).

    Returns:
        The agent's final answer as a string.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    create_kwargs = dict(model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto")
    if response_format:
        create_kwargs["response_format"] = response_format

    while True:
        response = client.chat.completions.create(**create_kwargs)
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_unset=True))
        create_kwargs["messages"] = messages

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"  [tool] {name}({args})")

            try:
                result = TOOL_DISPATCH[name](**args)
                content = json.dumps(result)
            except requests.HTTPError as e:
                content = json.dumps({"error": f"API error {e.response.status_code}: {e.response.text}"})
            except requests.RequestException as e:
                content = json.dumps({"error": str(e)})

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})


print("Agent ready.")

## 5. Live vs stale: side-by-side comparison

Now we ask **the same question** from Section 2 again — this time with tools enabled.

In [ ]:
print(f"Question: {question}")
print()
print("With live BuyWhere data:")
print("-" * 50)

answer_with_tools = run_price_agent(question)
print(answer_with_tools)

## 6. Structured price comparison table

For programmatic use, we can instruct the model to return a **machine-readable JSON table** using `response_format`. This is useful when you want to render results in a UI, sort by price, or feed into downstream logic.

In [ ]:
# JSON schema for a structured price comparison response
COMPARISON_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "price_comparison",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "product_query": {
                    "type": "string",
                    "description": "The product that was searched"
                },
                "listings": {
                    "type": "array",
                    "description": "All found listings sorted by price ascending",
                    "items": {
                        "type": "object",
                        "properties": {
                            "retailer": {"type": "string"},
                            "product_name": {"type": "string"},
                            "price_sgd": {"type": "number"},
                            "url": {"type": "string"}
                        },
                        "required": ["retailer", "product_name", "price_sgd", "url"],
                        "additionalProperties": False
                    }
                },
                "cheapest_retailer": {"type": "string"},
                "cheapest_price_sgd": {"type": "number"},
                "price_range_sgd": {
                    "type": "object",
                    "properties": {
                        "min": {"type": "number"},
                        "max": {"type": "number"}
                    },
                    "required": ["min", "max"],
                    "additionalProperties": False
                },
                "summary": {"type": "string"}
            },
            "required": ["product_query", "listings", "cheapest_retailer", "cheapest_price_sgd", "price_range_sgd", "summary"],
            "additionalProperties": False
        }
    }
}

print("Structured output schema defined.")

In [ ]:
query = "Samsung Galaxy S24 Ultra 256GB Singapore price comparison"
print(f"Query: {query}\n")

raw_json = run_price_agent(
    f"Compare prices for: {query}. Return all available listings sorted by price.",
    response_format=COMPARISON_SCHEMA,
)

comparison = json.loads(raw_json)

# Pretty-print the structured result
print(f"Product: {comparison['product_query']}")
print(f"Cheapest: SGD {comparison['cheapest_price_sgd']:.2f} at {comparison['cheapest_retailer']}")
print(f"Price range: SGD {comparison['price_range_sgd']['min']:.2f} – SGD {comparison['price_range_sgd']['max']:.2f}")
print()
print(f"{'Retailer':<20} {'Price (SGD)':>12}  Product")
print("-" * 70)
for listing in comparison["listings"]:
    print(f"{listing['retailer']:<20} {listing['price_sgd']:>12.2f}  {listing['product_name'][:40]}")
print()
print(f"Summary: {comparison['summary']}")

## 7. Batch comparison: multiple products at once

Need to compare multiple products simultaneously? OpenAI models can issue **parallel tool calls** in a single turn — the agent fetches all products at the same time and then synthesises a combined comparison.

In [ ]:
batch_query = """
I'm choosing between three laptops for university. Please compare current Singapore prices for:
1. MacBook Air M3 13-inch
2. Dell XPS 13
3. ASUS ZenBook 14

For each one tell me the cheapest available price and which retailer offers it.
"""

print("User query:", batch_query.strip())
print()
answer = run_price_agent(batch_query)
print("Agent answer:")
print("-" * 50)
print(answer)

## 8. Best-price finder

When the user just wants the single lowest price for a product, the `get_best_price_singapore` tool returns a direct answer without needing to sort through a list.

In [ ]:
query = "Where is the cheapest place to buy a Sony WH-1000XM5 in Singapore right now?"
print(f"User: {query}\n")

answer = run_price_agent(query)
print(f"Agent: {answer}")

## 9. Verifying grounding: does the model use live data?

A useful sanity check: inspect the messages list to confirm the model actually called the tool (rather than answering from training data). If `tool_calls` appears in the message history, the answer is grounded in live API data.

In [ ]:
def run_price_agent_with_trace(user_message: str) -> tuple[str, list[dict]]:
    """Same as run_price_agent but also returns the full message trace."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    while True:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto"
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_unset=True))

        if not message.tool_calls:
            return message.content, messages

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            try:
                result = TOOL_DISPATCH[name](**args)
                content = json.dumps(result)
            except requests.RequestException as e:
                content = json.dumps({"error": str(e)})
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})


answer, trace = run_price_agent_with_trace("What's the cheapest 65-inch OLED TV in Singapore?")

print("Grounding verification:")
print("-" * 50)
for msg in trace:
    role = msg.get("role", "?")
    if role == "assistant" and msg.get("tool_calls"):
        for tc in msg["tool_calls"]:
            print(f"  [GROUNDED] Tool called: {tc['function']['name']}({tc['function']['arguments'][:60]}...)")
    elif role == "tool":
        content_preview = msg["content"][:80].replace("\n", " ")
        print(f"  [LIVE DATA] Tool returned: {content_preview}...")

print()
print("Final answer:")
print(answer)

## Conclusion

### Key takeaways

| Without tools | With BuyWhere tools |
|---|---|
| Prices from training data (months/years old) | Prices fetched live at query time |
| Model may confidently hallucinate specific figures | Every figure is sourced from a real retailer listing |
| No retailer attribution | Specific retailer and link provided |
| Static — never updates | Always current |

### The grounding pattern

The core pattern from this recipe applies to any domain where freshness matters:

```
1. Identify the "stale knowledge" risk in your use case
2. Find a real-time API that covers that domain
3. Wrap it as an OpenAI function-calling tool
4. Tell the model in the system prompt to always call the tool first
5. (Optional) Use response_format for machine-readable structured output
```

The same pattern works for stock prices, weather, sports scores, flight availability, restaurant bookings — anywhere a model's training data goes stale.

### Next steps

- **Add price history** — combine BuyWhere with a local database to track whether a price is at an all-time low
- **Set up price alerts** — poll BuyWhere on a schedule and notify users when a target price is reached
- **Extend to other markets** — BuyWhere supports US retailers (Amazon, Best Buy, Walmart, Target) in beta
- **Build a UI** — wrap `run_price_agent` in a FastAPI endpoint and connect it to a chat interface

### Resources

- [BuyWhere API Documentation](https://buywhere.ai/docs/API_DOCUMENTATION)
- [BuyWhere free API key](https://buywhere.ai/api-keys)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)